# Figure 2: Nowcasting Feature Importance - Two-Layer Analysis

## Purpose
Analyze and visualize **feature importance for two-layer ensemble nowcasting** approach, separating structural patterns (Layer 1) from real-time shocks (Layer 2).

## Methodology
1. **Layer 1 (Forecasting Layer)**: Train XGBoost on lagged features (12+ month lags) for initial predictions
2. **Layer 2 (Residual Correction)**: Train XGBoost to predict residuals using real-time monthly indicators (`diff_set` features)
3. **SHAP Analysis**: Calculate feature importance separately for each layer
4. **Interpretation**:
   - **Layer 1 SHAP**: Long-term structural patterns (geography, climate trends, infrastructure)
   - **Layer 2 SHAP**: Short-term shocks and changes (monthly conflict events, price volatility, weather anomalies)
5. **Visualization**: Generate separate feature importance plots and variable group contribution charts

## Expected Outputs
- **shap_values_nowcasting_phase3_phase4_L1.jpg**: Top-10 features for Layer 1 (forecasting base)
- **shap_values_nowcasting_phase3_phase4_colored.jpg**: Top-10 features for Layer 2 (residual correction) with color-coded magnitudes
- **shap_values_nowcasting_bar_phase3_phase4_L1.jpg**: Stacked bar showing Layer 1 variable group contributions
- **shap_values_grouped_horizontal_bars_nowcasting.jpg**: Grouped Layer 2 variable contributions

## Key Insights
- **Layer 1** captures stable predictors with strong baseline signal
- **Layer 2** identifies rapid-response features critical for nowcasting accuracy
- Comparison reveals complementarity of structural vs. shock-driven patterns

## Performance Note
⚠️ **Two-layer SHAP computation requires 20-60 minutes** due to:
- Computing SHAP for both layers
- Larger feature sets (100+ features per layer)
- Two-stage model pipeline

## Note on Paths
**Execution note**:
1. Standalone: this notebook retrains both model layers from released source data before computing SHAP values.
2. The released code cells use package-relative data paths:

Current paths:
```
df_nowcasting = pd.read_csv(r'../1.Source Data/Nowcasting_Analysis_010825.csv')
df_forecasting = pd.read_csv(r'../1.Source Data/Forecasting_Analysis_010825.csv')
```

Equivalent construction:
```python
import os
data_dir = os.path.join('..', '1.Source Data')
df_nowcasting = pd.read_csv(os.path.join(data_dir, 'Nowcasting_Analysis_010825.csv'))
df_forecasting = pd.read_csv(os.path.join(data_dir, 'Forecasting_Analysis_010825.csv'))
```

3. Ensure hyperparameter JSON files exist:
   - `forecasting_hyperparameters.json`
   - `forecasting_hyperparameters_p3.json`

## Dependencies
- pandas, numpy, matplotlib, seaborn, xgboost, shap, scikit-learn
- Custom module: `food_crisis_functions`

In [ ]:
import numpy as np
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import mean_squared_error, accuracy_score, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import shap
from xgboost import XGBClassifier
# import random forest regressor
from sklearn.ensemble import RandomForestRegressor
#import linear regression
from sklearn.linear_model import LinearRegression
# import tqdm
from tqdm import tqdm
import tqdm
#import r2_score
from sklearn.metrics import r2_score
#import confusion matrix
from sklearn.metrics import confusion_matrix
# import roc auc score
from sklearn.metrics import roc_auc_score
from food_crisis_functions import *
import json
with open("forecasting_hyperparameters.json", "r") as file:
    best_params_xgb_regressor= json.load(file)
    
with open("forecasting_hyperparameters_p3.json", "r") as file:
    best_params_xgb_regressor_for_p3= json.load(file)

# read csv
df_nowcasting = pd.read_csv(r'../1.Source Data/Nowcasting_Analysis_010825.csv')
df_forecasting = pd.read_csv(r'../1.Source Data/Forecasting_Analysis_010825.csv')
#rename infra_index_m12 to infra_index_m12_l12, rename infra_index_s12 to infra_index_s12_l12
# add dummys for area_id and month year
#df = pd.concat([df, pd.get_dummies(df['area_id'], prefix='area_id')], axis=1)
#df = pd.concat([df, pd.get_dummies(df['date'], prefix='month_year')], axis=1)
# drop lat and lon
#df = df.drop(['lat', 'lon'], axis=1)
###drop fews_ipc_ha
#df = df.drop(['fews_ipc_ha'], axis=1)
# random split train and test
df_origin = df_forecasting.copy()
df = df_forecasting.copy()
y_pred_test = pd.DataFrame()
model_stats = pd.DataFrame()
#select phase1_percent is not na
df = df_origin[df_origin['phase1_percent'].notna()]
df_nowcasting = df_nowcasting[df_nowcasting['phase1_percent'].notna()]
# Sort by region and date
df = df.sort_values(by=['area_id', 'date'])
df_nowcasting = df_nowcasting.sort_values(by=['area_id', 'date'])
#drop overall phase
df = df.drop(['overall_phase'], axis=1)
#for each region, set last observation to be test set
# create a series of new outcome, phase2_worse=phase2_percent+phase3_percent+phase4_percent+phase5_percent, phase3_worse=phase3_percent+phase4_percent+phase5_percent, phase4_worse=phase4_percent+phase5_percent, phase5_worse=phase5_percent
df['phase2_worse'] = df['phase2_percent'] + df['phase3_percent'] + df['phase4_percent'] + df['phase5_percent']
df['phase3_worse'] = df['phase3_percent'] + df['phase4_percent'] + df['phase5_percent']
df['phase4_worse'] = df['phase4_percent'] + df['phase5_percent']
df['phase5_worse'] = df['phase5_percent']
#drop phase2_percent, phase3_percent, phase4_percent, phase5_percent, phase1_percent
df = df.drop(['phase2_percent', 'phase3_percent', 'phase4_percent', 'phase5_percent', 'phase1_percent'], axis=1)
# Splitting the data
#test_df = df.groupby('area_id').tail(1)
#train_df = df.drop(test_df.index)
#test_df = test_df.drop(['area_id','date'], axis=1)
#train_df = train_df.drop(['area_id','date'], axis=1)
y_pred_test = pd.DataFrame()
# drop anything after 2022-01-01
#df = df[df['date'] < '2021-01-01']
shape_values_df_ensemble_L1 = pd.DataFrame()
shape_values_df_ensemble_L2 = pd.DataFrame()
df_result = pd.DataFrame()
date = "2022-01-01"  # Define the 'date' variable
#order unique_dates
y_pred_test=pd.DataFrame()

diff_set = ['CPI',
 'fatalities_explosions_w5_m12',
 'GDP',
 'soil_moisture_mean_m12',
 'fatalities_explosions',
 'chirps_z_score',
 'EVI_m12',
 'event_count_battles_w10_m12',
 'event_count_battles_w5_m12',
 'price_index_s12',
 'event_count_explosions',
 'rainfall_chirps_s12',
 'event_count_violence_w10_m12',
 'WFP_Price_Change_s12',
 'WFP_Price_Volatility_m12',
 'WFP_Price_Change_m12',
 'infra_index_m12',
 'temperature_2m_mean_s12',
 'event_count_battles_m12',
 'fatalities_battles',
 'event_count_violence_m12',
 'event_count_explosions_m12',
 'fatalities_battles_w5_m12',
 'event_count_violence_w5_m12',
 'soil_moisture_mean_s12',
 'event_count_explosions_w5_m12',
 'WFP_Price_Volatility_s12',
 'event_count_violence',
 'precipitation_sum_m12',
 'event_count_violence_w10',
 'fatalities_violence_w10_m12',
 'event_count_battles',
 'event_count_explosions_w5',
 'fatalities_explosions_w10_m12',
 'event_count_battles_w10',
 'temperature_2m_mean_m12',
 'rainfall_chirps_m12',
 'fatalities_violence_m12',
 'WFP_Price_Change',
 'event_count_explosions_w10',
 'gini',
 'cpi',
 'fatalities_violence_w5_m12',
 'infra_index_s12',
 'shortwave_radiation_sum_m12',
 'price_index_m12',
 'fatalities_battles_w10_m12',
 'event_count_violence_w5',
 'fatalities_explosions_w10',
 'CC.PER.RNK',
 'fatalities_violence_w5',
 'shortwave_radiation_sum_s12',
 'fatalities_battles_w10',
 'fatalities_battles_m12',
 'nightlight_mean_m12',
 'fatalities_battles_w5',
 'event_count_battles_w5',
 'fatalities_violence_w10',
 'fatalities_violence',
 'infra_index',
 'fatalities_explosions_m12',
 'temperature_z_score',
 'event_count_explosions_w10_m12',
 'NY.GDP.PCAP.KD',
 'fatalities_explosions_w5',
 'precipitation_sum_s12',
 'WFP_Price_Volatility',
 'nightlight_mean_s12',
 'GOSIF_GPP_m12']



y_pred_test = pd.DataFrame()

for i in range(2, 6):
    train_df = df[df['date'] < date]
    test_df = df[df['date'] >= date]
    train_df = train_df.drop(['date','area_id'], axis=1)
    test_df = test_df.drop(['date','area_id'], axis=1)
    train_df_new = train_df.drop(['phase{}_worse'.format(j) for j in range(2, 6) if j != i], axis=1)
    test_df_new = test_df.drop(['phase{}_worse'.format(j) for j in range(2, 6) if j != i], axis=1)
    # drop rows with NaN in phase{}_percent
    train_df_new = train_df_new.dropna(subset=['phase{}_worse'.format(i)])
    test_df_new = test_df_new.dropna(subset=['phase{}_worse'.format(i)])
    test_index = test_df_new.index
    
    # LAYER 1: Predict outcome without diff_set features
    X_train_L1 = train_df_new.drop('phase{}_worse'.format(i), axis=1)
    y_train = train_df_new['phase{}_worse'.format(i)]
    X_test_L1 = test_df_new.drop('phase{}_worse'.format(i), axis=1)
    y_test = test_df_new['phase{}_worse'.format(i)]
    
    if i == 3:
        best_params_xgb_regressor = best_params_xgb_regressor_for_p3
    
    # Train Layer 1 model
    model_L1 = xgb.XGBRegressor(**best_params_xgb_regressor)
    model_L1.fit(X_train_L1, y_train)
    
    # Get Layer 1 predictions
    y_pred_L1 = model_L1.predict(X_test_L1)
    
    # Calculate residuals (errors) for training set
    X_train_full = train_df_new.drop('phase{}_worse'.format(i), axis=1)
    y_train_pred_L1 = model_L1.predict(X_train_L1)
    train_residuals = y_train - y_train_pred_L1
    
    # LAYER 2: Predict residuals using diff_set features
    # Extract only diff_set features for Layer 2
    train_now = df_nowcasting[df_nowcasting['date'] < date]
    test_now = df_nowcasting[df_nowcasting['date'] >= date]
    train_now = train_now.drop(['date','area_id'], axis=1)
    test_now = test_now.drop(['date','area_id'], axis=1)
    train_now_new = train_now.drop(['phase{}_percent'.format(j) for j in range(2, 6) if j != i], axis=1)
    test_now_new = test_now.drop(['phase{}_percent'.format(j) for j in range(2, 6) if j != i], axis=1)
    train_now_new = train_now_new.dropna(subset=['phase{}_percent'.format(i)])
    test_now_new = test_now_new.dropna(subset=['phase{}_percent'.format(i)])
    X_train_L2 = train_now_new[diff_set]
    X_test_L2 = test_now_new[diff_set]
    
    # Train Layer 2 model to predict residuals
    model_L2 = xgb.XGBRegressor(**best_params_xgb_regressor)  # You can optimize parameters for this layer too
    model_L2.fit(X_train_L2, train_residuals)
    
    # Get Layer 2 predictions (residual predictions)
    residual_pred = model_L2.predict(X_test_L2)
    
    # Final prediction = Layer 1 prediction + Layer 2 prediction (residuals)
    y_pred_final = y_pred_L1 + residual_pred
    
    # Store results
    y_pred_test = pd.concat([
        y_pred_test, 
        pd.DataFrame({
            'y_pred_L1': y_pred_L1,
            'residual_pred': residual_pred,
            'y_pred': y_pred_final,  # Combined prediction
            'y_test': y_test,
            'phase': [i]*len(y_pred_final),
            'test_index': test_index
        })
    ], ignore_index=True)
        # shap values
    explainer_L1 = shap.TreeExplainer(model_L1)
    shap_values_L1 = explainer_L1.shap_values(X_train_L1)
    
    #save shap values
    shap_values_df_L1 = pd.DataFrame(shap_values_L1, columns=X_train_L1.columns)
    
    # add a column to indicate the phase
    shap_values_df_L1['phase'] = i
    
    # append to shape_values_df_ensemble
    shape_values_df_ensemble_L1 = pd.concat([shape_values_df_ensemble_L1, shap_values_df_L1], ignore_index=True)
    
    explainer_L2 = shap.TreeExplainer(model_L2)
    shap_values_L2 = explainer_L2.shap_values(X_train_L2)
    
    #save shap values
    shap_values_df_L2 = pd.DataFrame(shap_values_L2, columns=X_train_L2.columns)
    
    # add a column to indicate the phase
    shap_values_df_L2['phase'] = i
    
    # append to shape_values_df_ensemble
    shape_values_df_ensemble_L2 = pd.concat([shape_values_df_ensemble_L2, shap_values_df_L2], ignore_index=True)

# Convert probabilities to phases
y_pred_test = convert_prob_to_phase(y_pred_test)
y_test = y_pred_test['overall_phase']
y_pred = y_pred_test['overall_phase_pred']

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Metrics
accuracy_score_new, sensitivity, precision, overall_r2 = all_metrics(y_test, y_pred, cm, y_pred_test)

print("Accuracy:", accuracy_score_new)
print("Sensitivity:", sensitivity)
print("Precision:", precision)
print("Overall R²(Phase 3 or more):", overall_r2)



In [ ]:
r2_frame_nowcasting = y_pred_test[['phase3_pred','phase3_test']]

In [ ]:
#rename phase3_pred to phase3_pred_nc, phase3_test to phase3_test_nc
r2_frame_nowcasting = r2_frame_nowcasting.rename(columns={'phase3_pred': 'phase3_pred_nc', 'phase3_test': 'phase3_test_nc'})
#save to csv
r2_frame_nowcasting.to_csv(r'produced_graph/r2_frame_nowcasting.csv', index=False)

In [ ]:
shap_values_df_ensemble_sum = shape_values_df_ensemble_L1.groupby('phase').sum()
#select phase != 2
shap_values_df_ensemble_sum = shap_values_df_ensemble_sum.drop([2], axis=0)
# take absolute value for each column
shap_values_df_ensemble_sum_abs = shap_values_df_ensemble_sum.abs()

# plot each phase in subplots, only plot the top 10 features
fig, axes = plt.subplots(2, 1, figsize=(10, 10))
axes = axes.flatten()

def rename_shap_feature_names(shap_values_df_ensemble_sum_abs, row_index=None):
    # Define the mapping dictionary
    feature_mapping = {
        'WFP_Price_Volatility': 'WFP Price Volatility',
        'lat': 'Latitude',
        'temperature_2m_mean_z_l12': '12 months lagged long-term surface temperature(z score)',
        'infra_index_m12': '12 months moving average infrastructure index',
        'CPI': 'CPI',
        'GOSIF_GPP_m12': '12 months moving average GOSIF-GPP',
        'area': 'Survey land area',
        'shortwave_radiation_sum_m12': '12 months moving average shortwave radiation',
        'aez_groupid_12000': 'Agricultural Ecological Zone',
        'GDP': 'GDP',
        'WFP_Price_Change_s12': 'WFP Price Change Percentage Standard Error in 12 months',
        'CC.PER.RNK': 'Control of Corruption',
        'temperature_2m_mean_l12': '12 months lagged short-term surface temperature(monthly)',
        'temperature_2m_mean_s12': 'surface temperature standard error in 12 months',
        'WFP_Price_Volatility_l12': '12 months lagged WF Price Volatility',
        'infra_index_m12_l12': '12 months lagged of 12months moving average infrastructure index',
        'lon': 'longitude',
        'WFP_Price_Change_l12': '12 months lagged WFP Price change Percentage',
        'estimated_population': 'estimated population',
        'EVI_l12': '12 months lagged EVI',
        'soil_moisture_mean_l12': '12 months lagged soil moisture',
        'infra_index_s12_l12': '12 months lagged infrastructure index standard error in 12 months',
        'nightlight_mean_l12': '12 months lagged nightlight index',
        'shortwave_radiation_sum_l12': '12 months lagged shortwave radiation',
        'aez_groupid_32000': 'Agricultural Ecological Zone',
        'phh2o_5-15cm_mean': 'Soil P.H.',
        'rainfall_chirps_z_l12': '12 months lagged long-term Ground Gauge rainfall (z score)',
        'infra_index_s12': 'Infrastructure index standard error in 12 months'
    }
    
    # If a specific row index is provided, rename only for that row
    if row_index is not None:
        # Get the top 10 features for the specified row
        top_features = shap_values_df_ensemble_sum_abs.iloc[row_index].sort_values(ascending=False)[:10]
        
        # Create a renamed Series with the mapped names
        renamed_features = top_features.copy()
        renamed_features.index = [feature_mapping.get(feature, feature) for feature in top_features.index]
        
        return renamed_features
    
    # Otherwise, apply to all rows (this would be more complex as it needs to handle the DataFrame structure)
    else:
        # Create a function to rename a single row's index
        def rename_row(row):
            sorted_row = row.sort_values(ascending=False)[:10]
            sorted_row.index = [feature_mapping.get(feature, feature) for feature in sorted_row.index]
            return sorted_row
        
        # Apply the function to each row
        renamed_df = shap_values_df_ensemble_sum_abs.apply(rename_row, axis=1)
        
        return renamed_df

# rename shap feature names for all rows
shap_values_df_ensemble_sum_abs = rename_shap_feature_names(shap_values_df_ensemble_sum_abs)
for i, ax in enumerate(axes):
    sns.barplot(x=shap_values_df_ensemble_sum_abs.iloc[i].sort_values(ascending=False)[:10], 
                y=shap_values_df_ensemble_sum_abs.iloc[i].sort_values(ascending=False)[:10].index, 
                ax=ax, color='blue')
    ax.set_title('Phase {} or higher'.format(i+3))
    ax.set_xlabel('SHAP Values')
    ax.set_ylabel('Features')
    
# add title - adjust position to avoid overlap with subplots
fig.text(0.5, -0.05, 'Top 10 Features with Highest SHAP Values for Each Phase in Nowcasting for Layer 1', 
         ha='center', fontsize=16)

# apply tight layout with adjusted parameters
plt.tight_layout(rect=[0, 0, 1, 0.95])  # leave room for the suptitle

# export to jpg with 300 dpi
plt.savefig('produced_graph/shap_values_nowcasting_phase3_phase4_L1.jpg', 
            dpi=300,
            format='jpeg',
            bbox_inches='tight',  # removes extra whitespace
            pil_kwargs={'compression': 'lzw'})  # adds LZW compression to reduce file size

# define meterological variables
# define meterological variables
geo_vars = ['elevation','ruggedness_index', 'slope','distance_to_river']


food_prices_vars = ['WFP_Price_Change_s12_l12','WFP_Price_Volatility_s12_l12', 'price_index_s12_l12', 'price_index_l12', 'WFP_Price_Change_l12','WFP_Price_Volatility_l12']
#rename WFP_Price_Change_s12_l12 to 

meteo_vars = ['shortwave_radiation_sum_l12',
 'temperature_2m_mean_l12',
 'rainfall_chirps_l12', 'precipitation_sum_l12', 'rainfall_chirps_z_s12_l12',
 'temperature_2m_mean_z_s12_l12','precipitation_sum_s12_l12', 'rainfall_chirps_z_l12',
 'temperature_2m_mean_z_l12', 'shortwave_radiation_sum_s12_l12',
 'temperature_2m_mean_s12_l12',
 'rainfall_chirps_s12_l12']

#define economic variables
econ_vars = ['estimated_population','area',
 'es_urban_pop',
 'urban_area',
 'nightlight_mean_s12_l12',
'nightlight_mean_l12',
 'NY.GDP.PCAP.KD_l12',
 'cpi_l12',
 'GDP_l12',
 'CPI_l12',
 'infra_index_l12',
 'CC.PER.RNK_l12',
 'gini_l12', 'infra_index_m12',
 'infra_index_s12', 'market_access',]

#define conflict variables
conflict_vars = [ 'fatalities_battles_l12',
 'fatalities_explosions_l12',
 'fatalities_violence_l12',
 'event_count_battles_l12',
 'event_count_explosions_l12',
 'event_count_violence_l12','fatalities_battles_w5_l12',
 'fatalities_explosions_w5_l12',
 'fatalities_violence_w5_l12',
 'event_count_battles_w5_l12',
 'event_count_explosions_w5_l12',
 'event_count_violence_w5_l12',
 'fatalities_battles_w10_l12',
 'fatalities_explosions_w10_l12',
 'fatalities_violence_w10_l12',
 'event_count_battles_w10_l12',
 'event_count_explosions_w10_l12',
 'event_count_violence_w10_l12','fatalities_battles_s12_l12',
 'fatalities_explosions_s12_l12',
 'fatalities_violence_s12_l12',
 'event_count_battles_s12_l12',
 'event_count_explosions_s12_l12',
 'event_count_violence_s12_l12', 'fatalities_battles_w5_s12_l12',
 'fatalities_explosions_w5_s12_l12',
 'fatalities_violence_w5_s12_l12',
 'event_count_battles_w5_s12_l12',
 'event_count_explosions_w5_s12_l12',
 'event_count_violence_w5_s12_l12',
 'fatalities_battles_w10_s12_l12',
 'fatalities_explosions_w10_s12_l12',
 'fatalities_violence_w10_s12_l12',
 'event_count_battles_w10_s12_l12',
 'event_count_explosions_w10_s12_l12',
 'event_count_violence_w10_s12_l12'
]

#define agricultural variables

agri_vars = ['aez_groupid_4000',
 'aez_groupid_7000',
 'aez_groupid_9000',
 'aez_groupid_10000',
 'aez_groupid_12000',
 'aez_groupid_17000',
 'aez_groupid_19000',
 'aez_groupid_25000',
 'aez_groupid_30000',
 'aez_groupid_31000',
 'aez_groupid_32000',
 'aez_groupid_33000',
 'aez_groupid_34000',
 'aez_groupid_36000',
 'aez_groupid_40000',
 'aez_groupid_43000','nitrogen_5-15cm_mean',
 'phh2o_5-15cm_mean',
 'cec_5-15cm_mean',
 'cfvo_5-15cm_mean',
 'soc_5-15cm_mean', 'soil_moisture_mean_l12',
 'EVI_l12',
 'GOSIF_GPP_l12',
 'soil_moisture_mean_s12_l12', 'EVI_s12_l12', 'GOSIF_GPP_s12_l12', 'cropland',
 'rangeland']
# group shap values by variable type, then plot four group of variables
shap_values_df_ensemble_sum_meteo = shap_values_df_ensemble_sum[meteo_vars]
shap_values_df_ensemble_sum_econ = shap_values_df_ensemble_sum[econ_vars]
shap_values_df_ensemble_sum_conflict = shap_values_df_ensemble_sum[conflict_vars]
shap_values_df_ensemble_sum_agri = shap_values_df_ensemble_sum[agri_vars]
shap_value_of_ensemble_sum_geo = shap_values_df_ensemble_sum[geo_vars]
shap_value_of_ensemble_sum_food_prices = shap_values_df_ensemble_sum[food_prices_vars]


# sum the same group of variables
shap_values_df_ensemble_sum_meteo = shap_values_df_ensemble_sum_meteo.sum(axis=1)
shap_values_df_ensemble_sum_econ = shap_values_df_ensemble_sum_econ.sum(axis=1)
shap_values_df_ensemble_sum_conflict = shap_values_df_ensemble_sum_conflict.sum(axis=1)
shap_values_df_ensemble_sum_agri = shap_values_df_ensemble_sum_agri.sum(axis=1)
shap_value_of_ensemble_sum_geo = shap_value_of_ensemble_sum_geo.sum(axis=1)
shap_value_of_ensemble_sum_food_prices = shap_value_of_ensemble_sum_food_prices.sum(axis=1)




# combine into a dataframe
shap_values_df_ensemble_sum_grouped = pd.concat([shap_values_df_ensemble_sum_meteo, 
                                                 shap_values_df_ensemble_sum_econ, shap_values_df_ensemble_sum_conflict, 
                                                 shap_values_df_ensemble_sum_agri, shap_value_of_ensemble_sum_geo, shap_value_of_ensemble_sum_food_prices], axis=1)
# phase
#shap_values_df_ensemble_sum_grouped['phase'] = [2,3,4,5]
shap_values_df_ensemble_sum_grouped['phase'] = [3,4,5]
#shap_values_df_ensemble_sum_grouped['phase'] = [2,3,4]
# rename columns
shap_values_df_ensemble_sum_grouped.columns = ['meteo', 'econ', 'conflict', 'agri', 'geo', 'food_prices','phase']
# drop phase column
shap_values_df_ensemble_sum_grouped = shap_values_df_ensemble_sum_grouped.drop('phase', axis=1)
# take absolute value, stacked bar chart, each bar is a phase
shap_values_df_ensemble_sum_grouped = shap_values_df_ensemble_sum_grouped.abs()

# normalize
shap_values_df_ensemble_sum_grouped = shap_values_df_ensemble_sum_grouped.div(shap_values_df_ensemble_sum_grouped.sum(axis=1), axis=0)

# rename phases, 3 to "phase 3 or higher", 4 to "phase 4 or higher"
shap_values_df_ensemble_sum_grouped.index = ['phase 3 or higher', 'phase 4 or higher', 'phase 5']


#remove phase 5
shap_values_df_ensemble_sum_grouped = shap_values_df_ensemble_sum_grouped.drop(["phase 5"], axis=0)
# plot stacked bar chart
# Create the stacked bar chart
ax = shap_values_df_ensemble_sum_grouped.plot.bar(
    stacked=True, 
    figsize=(10, 5)
    # Remove the title from here
)

# Make x-axis labels horizontal
plt.xticks(rotation=0)  # 0 degrees = horizontal

# Put legend in upper left corner
plt.legend(loc='upper left')

# Create space at the bottom for the title
plt.subplots_adjust(bottom=0.2)  # Adjust this value as needed

# Add the title at the bottom
plt.figtext(0.5, 0.01, 'Stacked Bar Chart of Shap Values Grouped by Variable Type in Nowcasting for Layer 1', 
            ha='center', fontsize=12)

# Save the figure with high resolution
plt.savefig(
    'produced_graph/shap_values_nowcasting_bar_phase3_phase4_L1.jpg', 
    dpi=300,
    format='jpeg',
    bbox_inches='tight',
    pil_kwargs={'compression': 'lzw'}
)

# Only show the figure after saving it
plt.show()

In [ ]:
shap_values_df_ensemble_sum = shape_values_df_ensemble_L2.groupby('phase').sum()
#select phase != 2
shap_values_df_ensemble_sum = shap_values_df_ensemble_sum.drop([2], axis=0)
# take absolute value for each column
shap_values_df_ensemble_sum_abs = shap_values_df_ensemble_sum.abs()


def rename_shap_feature_names(shap_values_df_ensemble_sum_abs, row_index=None):
    # Define the mapping dictionary
    feature_mapping = {
        'WFP_Price_Volatility': 'WFP Spatial Price Dispersion',
        'lat': 'Latitude',
        'temperature_2m_mean_z_l12': 'Long term Surface Temperature +',
        'infra_index_m12': 'Infrastructure Index *',
        'CPI': 'Current CPI',
        'GOSIF_GPP_m12': 'GOSIF-GPP *',
        'area': 'Land Area',
        'shortwave_radiation_sum_m12': 'Shortwave Radiation *',
        'aez_groupid_12000': 'Agricultural Ecological Zone:1',
        'GDP': 'Current GDP',
        'WFP_Price_Change_s12': 'WFP Price Change Percentage S.D. *',
        'CC.PER.RNK': 'Control of Corruption',
        'temperature_2m_mean_l12': 'Short term Surface Temperature +',
        'temperature_2m_mean_s12': 'Surface Temperature S.D. *',
        'WFP_Price_Volatility_l12': 'WFP Spatial Price Dispersion *',
        'infra_index_m12_l12': 'Infrastructure Index * +',
        'lon': 'Longitude',
        'WFP_Price_Change_l12': ' WFP Price change Percentage +',
        'estimated_population': 'Estimated Population',
        'EVI_l12': 'Enhanced Vegetation Index *',
        'soil_moisture_mean_l12': 'Soil Moisture +',
        'infra_index_s12_l12': 'Infrastructure Index S.D. + *',
        'nightlight_mean_l12': 'VIIRS Nightlight +',
        'shortwave_radiation_sum_l12': 'Shortwave Radiation +',
        'aez_groupid_32000': 'Agricultural Ecological Zone:2',
        'phh2o_5-15cm_mean': 'Soil P.H.',
        'rainfall_chirps_z_l12': 'Long-term Ground Gauge rainfall +',
        'infra_index_s12': 'Infrastructure Index S.D. *',
        'nightlight_mean_m12': 'VIIRS Nightlight *',
        'nightlight_mean_s12': 'VIIRS Nightlight S.D. *',
        'WFP_Price_Change_m12': 'WFP Price Change *',
        'soil_moisture_mean_m12': 'Soil Moisture *',
        'fatalities_battles_w5': 'Fatalities in battles in 5 stations',
        'NY.GDP.PCAP.KD': 'GDP per capita',
        'event_count_battles_w10_m12': 'Event Count in battles in 10 stations *',
        'WFP_Price_Volatility_m12': 'WFP Spatial Price Dispersion *',
        'shortwave_radiation_sum_s12': 'Shortwave Radiation S.D. *',
        'fatalities_battles': 'Fatalities in battles',
        'EVI_m12': 'Enhanced Vegetation Index *'
    }
    
    # If a specific row index is provided, rename only for that row
    if row_index is not None:
        # Get the top 10 features for the specified row
        top_features = shap_values_df_ensemble_sum_abs.iloc[row_index].sort_values(ascending=False)[:10]
        
        # Create a renamed Series with the mapped names
        renamed_features = top_features.copy()
        renamed_features.index = [feature_mapping.get(feature, feature) for feature in top_features.index]
        
        return renamed_features
    
    # Otherwise, apply to all rows (this would be more complex as it needs to handle the DataFrame structure)
    else:
        # Create a function to rename a single row's index
        def rename_row(row):
            sorted_row = row.sort_values(ascending=False)[:10]
            sorted_row.index = [feature_mapping.get(feature, feature) for feature in sorted_row.index]
            return sorted_row
        
        # Apply the function to each row
        renamed_df = shap_values_df_ensemble_sum_abs.apply(rename_row, axis=1)
        
        return renamed_df
    
    

food_prices_vars = ['WFP_Price_Change', 'WFP_Price_Volatility', 'price_index_m12', 'WFP_Price_Change_m12', 
'WFP_Price_Volatility_m12', 'price_index_s12', 'WFP_Price_Change_s12', 'WFP_Price_Volatility_s12']
#rename WFP_Price_Change_s12_l12 to 

weather_vars = ['precipitation_sum_m12', 'shortwave_radiation_sum_m12', 'temperature_2m_mean_m12', 
'rainfall_chirps_m12', 'precipitation_sum_s12', 'shortwave_radiation_sum_s12', 
'temperature_2m_mean_s12', 'rainfall_chirps_s12', 'chirps_z_score', 'temperature_z_score']

#define economic variables
econ_vars = ['CPI', 'GDP', 'infra_index', 'infra_index_m12', 'infra_index_s12', 
'nightlight_mean_m12', 'nightlight_mean_s12', 'NY.GDP.PCAP.KD']

#define conflict variables
conflict_vars = ['fatalities_battles', 'fatalities_explosions', 'fatalities_violence', 
'event_count_battles', 'event_count_explosions', 'event_count_violence', 
'fatalities_battles_w5', 'fatalities_explosions_w5', 'fatalities_violence_w5', 
'event_count_battles_w5', 'event_count_explosions_w5', 'event_count_violence_w5', 
'fatalities_battles_w10', 'fatalities_explosions_w10', 'fatalities_violence_w10', 
'event_count_battles_w10', 'event_count_explosions_w10', 'event_count_violence_w10', 
'fatalities_battles_m12', 'fatalities_explosions_m12', 'fatalities_violence_m12', 
'event_count_battles_m12', 'event_count_explosions_m12', 'event_count_violence_m12', 
'fatalities_battles_w5_m12', 'fatalities_explosions_w5_m12', 'fatalities_violence_w5_m12', 
'event_count_battles_w5_m12', 'event_count_explosions_w5_m12', 'event_count_violence_w5_m12', 
'fatalities_battles_w10_m12', 'fatalities_explosions_w10_m12', 'fatalities_violence_w10_m12', 
'event_count_battles_w10_m12', 'event_count_explosions_w10_m12', 'event_count_violence_w10_m12']

#define agricultural variables

agri_vars = ['GOSIF_GPP_m12', 'soil_moisture_mean_m12', 'soil_moisture_mean_s12', 'EVI_m12']
# group shap values by variable type, then plot four group of variables
shap_values_df_ensemble_sum_weather = shap_values_df_ensemble_sum[weather_vars]
shap_values_df_ensemble_sum_econ = shap_values_df_ensemble_sum[econ_vars]
shap_values_df_ensemble_sum_conflict = shap_values_df_ensemble_sum[conflict_vars]
shap_values_df_ensemble_sum_agri = shap_values_df_ensemble_sum[agri_vars]
shap_value_of_ensemble_sum_food_prices = shap_values_df_ensemble_sum[food_prices_vars]


# sum the same group of variables
shap_values_df_ensemble_sum_weather = shap_values_df_ensemble_sum_weather.sum(axis=1)
shap_values_df_ensemble_sum_econ = shap_values_df_ensemble_sum_econ.sum(axis=1)
shap_values_df_ensemble_sum_conflict = shap_values_df_ensemble_sum_conflict.sum(axis=1)
shap_values_df_ensemble_sum_agri = shap_values_df_ensemble_sum_agri.sum(axis=1)
shap_value_of_ensemble_sum_food_prices = shap_value_of_ensemble_sum_food_prices.sum(axis=1)




# combine into a dataframe
shap_values_df_ensemble_sum_grouped = pd.concat([shap_values_df_ensemble_sum_weather, 
                                                 shap_values_df_ensemble_sum_econ, shap_values_df_ensemble_sum_conflict, 
                                                 shap_values_df_ensemble_sum_agri, shap_value_of_ensemble_sum_food_prices], axis=1)
# phase
#shap_values_df_ensemble_sum_grouped['phase'] = [2,3,4,5]
shap_values_df_ensemble_sum_grouped['phase'] = [3,4,5]
#shap_values_df_ensemble_sum_grouped['phase'] = [2,3,4]
# rename columns
shap_values_df_ensemble_sum_grouped.columns = ['Weather', 'Econ', 'Conflict', 'Agri', 'Food Prices','Phase']
# drop phase column
shap_values_df_ensemble_sum_grouped = shap_values_df_ensemble_sum_grouped.drop('Phase', axis=1)
# take absolute value, stacked bar chart, each bar is a phase
shap_values_df_ensemble_sum_grouped = shap_values_df_ensemble_sum_grouped.abs()

# normalize
shap_values_df_ensemble_sum_grouped = shap_values_df_ensemble_sum_grouped.div(shap_values_df_ensemble_sum_grouped.sum(axis=1), axis=0)

# rename phases, 3 to "phase 3 or higher", 4 to "phase 4 or higher"
shap_values_df_ensemble_sum_grouped.index = ['phase 3 or higher', 'phase 4 or higher', 'phase 5']


#remove phase 5
shap_values_df_ensemble_sum_grouped = shap_values_df_ensemble_sum_grouped.drop(["phase 5"], axis=0)

In [ ]:
def get_color_for_value(value):
    if value <= 1:
        return '#A1D99B'
    elif value <= 5:
        return '#74C476'
    elif value <= 10:
        return '#41AB5D'
    else:
        return '#006D2C'
    
def get_color_for_value_1(value):
    if value <= 1:
        return "#5BADD9"
    elif value <= 5:
        return "#4A85C4"
    elif value <= 10:
        return "#2668C5"
    else:
        return "#3A2FD5"

# Plotting the first graph with color coding
fig, axes = plt.subplots(2, 1, figsize=(10, 10))
axes = axes.flatten()

# rename shap feature names for all rows
shap_values_df_ensemble_sum_abs = rename_shap_feature_names(shap_values_df_ensemble_sum_abs)

for i, ax in enumerate(axes):
    # Get the top 10 features
    top_features = shap_values_df_ensemble_sum_abs.iloc[i].sort_values(ascending=False)[:10]

    #reverse order of top_features
    top_features = top_features[::-1]
    
    # Create a list of colors based on SHAP values
    if i == 0:
        colors = [get_color_for_value(value) for value in top_features]
    else:
        colors = [get_color_for_value_1(value) for value in top_features]
    # Create the bar plot with custom colors
    bars = ax.barh(y=top_features.index, width=top_features, color=colors)
    ax.set_yticklabels(top_features.index,fontsize=15)
    ax.set_title('Phase {} or higher'.format(i+3),fontsize=15)
    ax.set_xlabel('SHAP Values',fontsize=15)
    ax.set_ylabel('Features',fontsize=15) # The y-axis labels (features) might contain '+' or '*'

    # ================== START MINIMAL EDIT ==================
    # Add a legend for color meaning AND symbol meaning
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D # Import Line2D for dummy handles

    # Original legend elements for SHAP value colors
    shap_legend_elements_1 = [
        Patch(facecolor='#A1D99B', edgecolor='grey', label='0-1'),  # More Prominent Light Green
        Patch(facecolor='#74C476', edgecolor='grey', label='1-5'),  # Medium Green
        Patch(facecolor='#41AB5D', edgecolor='grey', label='5-10'), # Dark Green
        Patch(facecolor='#006D2C', edgecolor='grey', label='>10')   # Darkest Green
    ]
    shap_legend_elements_2 = [
        Patch(facecolor="#5BADD9", edgecolor='grey', label='0-1'),  # More Prominent Light Blue
        Patch(facecolor="#4A85C4", edgecolor='grey', label='1-5'),  # Medium Blue
        Patch(facecolor="#2668C5", edgecolor='grey', label='5-10'), # Dark Blue
        Patch(facecolor="#3A2FD5", edgecolor='grey', label='>10')   # Darkest Blue
    ]
    shap_legend_elements = shap_legend_elements_1 if i == 0 else shap_legend_elements_2
    shap_labels = [h.get_label() for h in shap_legend_elements]

    # Dummy handles and labels for the variable type indicators
    symbol_handles = [
        Line2D([0], [0], marker=None, color='None', linestyle='None'), # Invisible handle
        Line2D([0], [0], marker=None, color='None', linestyle='None')  # Invisible handle
    ]
    symbol_labels = ["'+': lagged 12 months", "'*': 12-month moving average"]

    # Combine all handles and labels
    all_handles = shap_legend_elements
    all_labels = shap_labels

    # Add the combined legend
    # Grouped titles can be complex; let's use one main title. Adjust loc if needed.
    ax.legend(handles=all_handles, labels=all_labels, title="Legend", loc="lower right", fontsize=12) # Added fontsize adjustment
    
# add title under the figure, use bold font
fig.text(0.5, -0.05, '(B) Top 10 Features with Highest SHAP Values for Each Phases in 2nd Layer Nowcasting',ha='center', fontsize=16, fontweight='bold')

# apply tight layout with adjusted parameters
plt.tight_layout(rect=[0, 0, 1, 0.95])  # leave room for the suptitle

# export to jpg with 300 dpi
plt.savefig('produced_graph/shap_values_nowcasting_phase3_phase4_colored.jpg', 
            dpi=100,
            format='jpeg',
            bbox_inches='tight',  # removes extra whitespace
            pil_kwargs={'compression': 'lzw'})  # adds LZW compression to reduce file size


In [ ]:

# For the second graph - Creating horizontal bar chart grouped by variable type
# First, let's transpose the data to get the format we need
shap_grouped_transposed = shap_values_df_ensemble_sum_grouped.T

# Create a figure with appropriate size
fig, ax = plt.subplots(figsize=(12, 8))

# Set width of bars
bar_width = 0.35
index = np.arange(len(shap_grouped_transposed))

# Get the values for each phase
phase3_values = shap_grouped_transposed['phase 3 or higher']
phase4_values = shap_grouped_transposed['phase 4 or higher']

# Use distinctive colors for each phase
phase3_color = '#006D2C'  # green
phase4_color = '#3A2FD5'  # blue

# Creating the grouped horizontal bar chart
bars1 = ax.barh(index + bar_width/2, phase3_values, bar_width, color=phase3_color, label='Phase 3 or higher', edgecolor='black', linewidth=0.5)
bars2 = ax.barh(index - bar_width/2, phase4_values, bar_width, color=phase4_color, label='Phase 4 or higher', edgecolor='black', linewidth=0.5)

# Add labels, title, and legend
ax.set_xlabel('Normalized SHAP Value Contribution',fontsize=15)
ax.set_yticks(index)
ax.set_yticklabels(shap_grouped_transposed.index,fontsize=15)
fig.text(0.5, -0.05, '(D) Variable Group Contributions to Food Insecurity Phases in 2nd Layer Nowcasting', 
         ha='center', fontsize=15,fontweight='bold')
ax.legend(loc='lower right',fontsize=13)

# Apply tight layout
plt.tight_layout()

# Export to jpg with 300 dpi
plt.savefig('produced_graph/shap_values_grouped_horizontal_bars_nowcasting.jpg', 
            dpi=100,
            format='jpeg',
            bbox_inches='tight',
            pil_kwargs={'compression': 'lzw'})

plt.show()